<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/Panel_Data_ATR_LightGBM_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Panel Data、ATR 動態停利停損以及 LightGBM 樹狀機器學習演算法
import datetime
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings('ignore')

try:
  from IPython.display import display
except ImportError:

  def display(x):
    print(x)


# 股票池定義
stock_dict = {
    '0050.TW': '元大台灣50',
    '0056.TW': '元大高股息',
    '00878.TW': '國泰永續高股息',
    '00770.TW': '國泰北美科技',
    '00981A.TW': '統一台股增長主動式',
    'AAPL': 'Apple 蘋果',
    'GOOG': 'Google / Alphabet',
    'META': 'Meta',
    'MSFT': 'Microsoft 微軟',
    'NVDA': 'NVIDIA 輝達',
    'TSM': '台積電 ADR',
    'TSLA': 'Tesla 特斯拉',
    'ENTG': 'Entegris 英特格',
    'SMR': 'NuScale Power 小型核反應爐',
    'BE': 'Bloom Energy 燃料電池',
    'JNJ': 'Johnson & Johnson 嬌生',
    'SPCX': 'SPACs ETF',
    '2382.TW': '廣達',
    '3231.TW': '緯創',
    '6669.TW': '緯穎',
    '2317.TW': '鴻海',
    '2356.TW': '英業達',
    '2324.TW': '仁寶',
    '2376.TW': '技嘉',
    '3706.TW': '神達',
    '2377.TW': '微星',
    '2357.TW': '華碩',
    '4938.TW': '和碩',
    '3005.TW': '神基',
    '5274.TWO': '信驊',
    '3689.TWO': '湧德',
    '3357.TWO': '臺慶科',
    '6862.TW': '三集瑞-KY',
    '6821.TWO': '聯寶',
    '3207.TWO': '耀勝',
    '6197.TW': '佳必琪',
    '8103.TW': '瀚荃',
    '3526.TWO': '凡甲',
    '3605.TW': '宏致',
    '2345.TW': '智邦',
    '5388.TW': '中磊',
    '3558.TWO': '神準',
    '3704.TW': '合勤控',
    '4906.TW': '正文',
    '8210.TW': '勤誠',
    '6117.TW': '迎廣',
    '6235.TW': '華孚',
    '2354.TW': '鴻準',
    '3376.TW': '新日興',
    '3548.TWO': '兆利',
    '5243.TW': '乙盛-KY',
    '4966.TWO': '譜瑞-KY',
    '5269.TW': '祥碩',
    '6104.TWO': '創唯',
    '6756.TW': '威鋒電子',
    '6715.TW': '嘉基',
    '3533.TW': '嘉澤',
    '3217.TWO': '優群',
    '3023.TW': '信邦',
    '2392.TW': '正崴',
    '3035.TW': '智原',
    '6643.TWO': 'M31',
    '2308.TW': '台達電',
    '2301.TW': '光寶科',
    '6282.TW': '康舒',
    '6412.TW': '群電',
    '3665.TW': '貿聯-KY',
    '3017.TW': '奇鋐',
    '3324.TWO': '雙鴻',
    '3653.TW': '健策',
    '2421.TW': '建準',
    '8996.TW': '高力',
    '3483.TWO': '力致',
    '6230.TW': '尼得科超眾',
    '3013.TW': '晟銘電',
    '6805.TW': '富世達',
    '6781.TW': 'AES-KY',
    '3211.TWO': '順達',
    '6121.TWO': '新普',
    '3323.TWO': '加百裕',
    '3625.TWO': '西勝',
    '8038.TWO': '長園科',
    '4931.TWO': '新盛力',
    '1519.TW': '華城',
    '1513.TW': '中興電',
    '1514.TW': '亞力',
    '1503.TW': '士電',
    '1609.TW': '大亞',
    '1605.TW': '華新',
    '1608.TW': '華榮',
    '6869.TW': '雲豹能源',
    '3037.TW': '欣興',
    '8046.TW': '南電',
    '3189.TW': '景碩',
    '4958.TW': '臻鼎-KY',
    '2368.TW': '金像電',
    '3044.TW': '健鼎',
    '2313.TW': '華通',
    '8155.TWO': '博智',
    '2383.TW': '台光電',
    '6274.TWO': '台燿',
    '6213.TW': '聯茂',
    '2344.TW': '華邦電',
    '2408.TW': '南亞科',
    '2337.TW': '旺宏',
    '3006.TW': '晶豪科',
    '3260.TWO': '威剛',
    '2451.TW': '創見',
    '4967.TW': '十銓',
    '8271.TW': '宇瞻',
    '5289.TWO': '宜晶',
    '8299.TWO': '群聯',
    '5351.TWO': '鈺創',
    '4979.TWO': '華星光',
    '6442.TW': '光聖',
    '4908.TWO': '前鼎',
    '3163.TWO': '波若威',
    '3450.TW': '聯鈞',
    '6426.TW': '統新',
    '4977.TW': '眾達-KY',
    '6530.TWO': '創威',
    '3363.TWO': '上詮',
    '3234.TWO': '光環',
    '4903.TWO': '聯光通',
    '3081.TWO': '聯亞',
    '4991.TWO': '環宇-KY',
    '4971.TWO': 'IET-KY',
    '6588.TWO': '東典光電',
    '3491.TWO': '昇達科',
    '2314.TW': '台揚',
    '6285.TW': '啟碁',
    '3105.TWO': '穩懋',
    '2455.TW': '全新',
    '3138.TW': '耀登',
    '2419.TW': '仲琦',
    '2327.TW': '國巨',
    '2492.TW': '華新科',
    '2375.TW': '凱美',
    '2478.TW': '大毅',
    '3026.TW': '禾伸堂',
    '3090.TW': '日電貿',
    '6173.TWO': '信昌電',
    '6155.TW': '鈞寶',
    '6175.TWO': '立敦',
    '5328.TWO': '華容',
    '3236.TWO': '千如',
    '2049.TW': '上銀',
    '4576.TW': '大銀微系統',
    '4585.TW': '達明',
    '2359.TW': '所羅門',
    '6188.TWO': '廣明',
    '8374.TW': '羅昇',
    '5443.TWO': '均豪',
    '6640.TWO': '均華',
    '2464.TW': '盟立',
    '6215.TW': '和椿',
    '4562.TW': '穎漢',
    '1590.TW': '亞德客-KY',
    '1504.TW': '東元',
    '3711.TW': '日月光投控',
    '2449.TW': '京元電子',
    '6257.TW': '矽格',
    '3264.TWO': '欣銓',
    '6239.TW': '力成',
    '2329.TW': '華泰',
    '2441.TW': '超豐',
    '3131.TWO': '弘塑',
    '3583.TW': '辛耘',
    '6187.TWO': '萬潤',
    '2467.TW': '志聖',
    '8027.TWO': '钛昇',
    '3481.TW': '群創',
    '2409.TW': '友達',
    '5434.TW': '崇越',
    '3010.TW': '華立',
    '1560.TW': '中砂',
    '3680.TWO': '家登',
    '5234.TW': '達興材料',
    '4749.TWO': '新應材',
    '8028.TW': '昇陽半導體',
    '6515.TW': '穎崴',
    '6683.TWO': '雍智科技',
    '6510.TWO': '精測',
    '6223.TWO': '旺矽',
    '2330.TW': '台積電',
    '2303.TW': '聯電',
    '2454.TW': '聯發科',
    '3034.TW': '聯詠',
    '3661.TW': '世芯-KY',
    '3443.TW': '創意',
    '4961.TW': '天鈺',
    '6415.TW': '矽力-KY',
    '6531.TW': '愛普*',
    '2404.TW': '漢唐',
    '1773.TW': '勝一',
    '3008.TW': '大立光',
    '4915.TW': '先進光',
    '5288.TW': '匯鑽科',
    '3644.TWO': '凌嘉科',
    '7769.TW': '鴻勁',
    '1303.TW': '南亞',
    '2395.TW': '研華',
    '6166.TW': '凌華',
    '8050.TWO': '廣積',
    '3556.TWO': '禾瑞亞',
    '2414.TW': '精技',
    '6414.TW': '樺漢',
    '3022.TW': '威強電',
    '2397.TW': '友通',
    '5314.TWO': '世紀',
    '2393.TW': '億光',
    '2465.TW': '麗臺',
    '5536.TWO': '聖暉*',
    '2201.TW': '裕隆',
    '2204.TW': '中華',
    '2206.TW': '三陽工業',
    '1536.TW': '和大',
    '2231.TW': '聯嘉',
    '3552.TWO': '同致',
    '6279.TWO': '胡連',
    '2353.TW': '宏碁',
    '8163.TW': '達方',
    '8043.TWO': '蜜望實',
    '2892.TW': '第一金',
    '5880.TW': '合庫金',
    '1210.TW': '大成',
    '1215.TW': '卜蜂',
    '1216.TW': '統一',
    '2912.TW': '統一超',
    '5903.TWO': '全家',
    '6770.TW': '力積電',
    '2342.TW': '茂矽',
    '3707.TWO': '漢磊',
    '3016.TW': '嘉晶',
    '6196.TW': '帆宣',
    '6139.TW': '亞翔',
    '6613.TWO': '朋億*',
    '4755.TW': '三福化',
    '4768.TWO': '晶呈科技',
    '3563.TW': '牧德',
    '1717.TW': '長興',
    '1815.TWO': '富喬',
    '1802.TW': '台玻',
    '5340.TWO': '建榮',
    '5475.TWO': '德宏',
    '3167.TW': '大量',
    '6438.TW': '迅得',
    '1595.TWO': '川寶',
    '6147.TWO': '頎邦',
    '8150.TW': '南茂',
    '6552.TW': '易華電',
    '3305.TW': '昇貿',
    '3631.TWO': '晟楠',
    '2059.TW': '川湖',
    '6584.TWO': '南俊國際',
    '8358.TWO': '金居',
    '8021.TW': '尖點',
    '6672.TW': '騰輝電子-KY',
}


# 1. 改良版標籤函數：使用 ATR 動態停利停損
def apply_labeling_atr(df, horizon=10):
  closes = df['Close'].values
  highs = df['High'].values
  lows = df['Low'].values
  labels = np.zeros(len(df))

  high_low = highs - lows
  high_cp = np.abs(highs - np.roll(closes, 1))
  low_cp = np.abs(lows - np.roll(closes, 1))
  tr = np.maximum(high_low, np.maximum(high_cp, low_cp))
  atr = pd.Series(tr).rolling(14).mean().values

  for i in range(len(df) - horizon):
    entry_p = closes[i]
    if np.isnan(atr[i]) or entry_p == 0:
      continue

    upper_target = entry_p + (2.0 * atr[i])
    lower_target = entry_p - (1.5 * atr[i])

    hit = 0
    for step in range(1, horizon + 1):
      if lows[i + step] <= lower_target:
        break
      if highs[i + step] >= upper_target:
        hit = 1
        break
    labels[i] = hit

  # 未滿 horizon 天的最新資料設為 NaN
  labels[-horizon:] = np.nan
  df['Target'] = labels
  return df


# 2. 改良版特徵工程：修正大盤 Alpha 算術順序
def compute_panel_features(df, market_df):
  d = df.copy()

  ma5 = d['Close'].rolling(5).mean()
  ma20 = d['Close'].rolling(20).mean()

  vol_20 = d['Volume'].rolling(20).mean()
  d['Vol_Ratio_Self'] = d['Volume'] / (vol_20 + 1e-6)
  d['BIAS_20'] = (d['Close'] - ma20) / (ma20 + 1e-6)

  ma10 = d['Close'].rolling(10).mean()
  d['MA_Tangle'] = (
      pd.concat([ma5, ma10, ma20], axis=1).max(axis=1)
      - pd.concat([ma5, ma10, ma20], axis=1).min(axis=1)
  ) / (ma20 + 1e-6)

  d['Historical_Volatility'] = (
      d['Close'].pct_change().rolling(60).std() * np.sqrt(252)
  )

  m_close = (
      market_df['Close'].iloc[:, 0]
      if isinstance(market_df['Close'], pd.DataFrame)
      else market_df['Close']
  )
  m_ret_5 = m_close.pct_change(5).reindex(d.index, method='ffill')
  d['Mkt_Alpha'] = d['Close'].pct_change(5) - m_ret_5

  cols = [
      'Vol_Ratio_Self',
      'BIAS_20',
      'MA_Tangle',
      'Historical_Volatility',
      'Mkt_Alpha',
  ]
  return d, cols


# 3. 下載數據
print('正在下載台股 (^TWII) 與美股 (^GSPC) 數據...')
tw_market = yf.download('^TWII', period='4y', progress=False)
us_market = yf.download('^GSPC', period='4y', progress=False)

if isinstance(tw_market.columns, pd.MultiIndex):
  tw_market = tw_market.xs('^TWII', axis=1, level=1)
if isinstance(us_market.columns, pd.MultiIndex):
  us_market = us_market.xs('^GSPC', axis=1, level=1)

print('正在批次下載個股數據...')
all_tickers = list(stock_dict.keys())
stocks_data = yf.download(
    all_tickers, period='4y', group_by='ticker', threads=True, progress=False
)

# 4. 建立 Panel Data
print('正在建構 Panel Data...')
panel_dfs = []

for ticker, name in stock_dict.items():
  try:
    if len(all_tickers) == 1:
      df = stocks_data.copy()
    else:
      df = stocks_data[ticker].dropna(how='all').copy()

    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    if len(df) < 250:
      continue

    market_df = (
        tw_market
        if (ticker.endswith('.TW') or ticker.endswith('.TWO'))
        else us_market
    )

    df = apply_labeling_atr(df)
    df, f_cols = compute_panel_features(df, market_df)

    df['Ticker'] = ticker
    df['Name'] = name

    df = df.dropna(subset=f_cols)
    panel_dfs.append(df)
  except Exception:
    continue

if panel_dfs:
  full_panel_df = pd.concat(panel_dfs)

  labeled_panel_df = full_panel_df.dropna(subset=['Target'])

  split_date = pd.to_datetime('2026-03-31')
  train_end_date = split_date - pd.Timedelta(days=15)
  test_end_date = pd.to_datetime('2026-06-30')

  train_data = labeled_panel_df[labeled_panel_df.index <= train_end_date]
  test_data = labeled_panel_df[
      (labeled_panel_df.index > split_date)
      & (labeled_panel_df.index <= test_end_date)
  ]

  print(
      f'\n全市場 Panel 訓練集筆數: {len(train_data)} | 測試集筆數:'
      f' {len(test_data)}'
  )

  # 5. 訓練模型
  print('正在訓練全市場通用 LightGBM 模型...')
  model = lgb.LGBMClassifier(
      n_estimators=200,
      learning_rate=0.03,
      max_depth=4,
      subsample=0.8,
      colsample_bytree=0.8,
      random_state=42,
      verbose=-1,
  )
  model.fit(train_data[f_cols], train_data['Target'])

  # 6. 回測驗證與全方位績效評估 (新增勝率<52%及整體歷史指標)
  if not test_data.empty:
    test_data = test_data.copy()
    test_data['Prob'] = model.predict_proba(test_data[f_cols])[:, 1]
    threshold = 0.52

    # 劃分高勝率組與低勝率組
    high_prob_signals = test_data[test_data['Prob'] >= threshold].sort_values(
        by='Prob', ascending=False
    )
    low_prob_signals = test_data[test_data['Prob'] < threshold].sort_values(
        by='Prob', ascending=False
    )

    # 1. 門檻 >= 52% (高勝率組) 評估
    print(
        f'\n=== Panel Data 模型回測結果 (高勝率組：預測機率 >='
        f' {int(threshold*100)}%) ==='
    )
    if not high_prob_signals.empty:
      high_count = len(high_prob_signals)
      high_success = int(high_prob_signals['Target'].sum())
      high_win_rate = (high_success / high_count) * 100
      print(
          f'高勝率組訊號數: {high_count} | 成功數: {high_success} | 門檻內勝率'
          f' (Precision): {high_win_rate:.2f}%'
      )
      display(high_prob_signals[['Ticker', 'Name', 'Target', 'Prob']].head(10))
    else:
      print('無符合高勝率門檻之訊號')
      high_win_rate = np.nan

    # 2. 門檻 < 52% (低勝率組) 評估
    print(
        f'\n=== Panel Data 模型回測結果 (低勝率組：預測機率 <'
        f' {int(threshold*100)}%) ==='
    )
    if not low_prob_signals.empty:
      low_count = len(low_prob_signals)
      low_success = int(low_prob_signals['Target'].sum())
      low_win_rate = (low_success / low_count) * 100
      print(
          f'低勝率組訊號數: {low_count} | 成功數: {low_success} | 門檻外勝率:'
          f' {low_win_rate:.2f}%'
      )
      display(low_prob_signals[['Ticker', 'Name', 'Target', 'Prob']].head(10))
    else:
      print('無低於門檻之資料')
      low_win_rate = np.nan

    # 3. 全體歷史自然勝率與模型全體分類指標
    y_true = test_data['Target'].values
    y_pred = (test_data['Prob'] >= threshold).astype(int).values
    y_prob = test_data['Prob'].values

    total_samples = len(test_data)
    total_success = int(y_true.sum())
    baseline_win_rate = (total_success / total_samples) * 100

    accuracy = accuracy_score(y_true, y_pred) * 100
    precision = precision_score(y_true, y_pred, zero_division=0) * 100
    recall = recall_score(y_true, y_pred, zero_division=0) * 100
    try:
      auc = roc_auc_score(y_true, y_prob)
    except Exception:
      auc = np.nan

    print('\n=== 全市場測試集整體統計與模型分類效能 ===')
    summary_metrics = pd.DataFrame([{
        '測試集總樣本數': total_samples,
        '歷史自然勝率 (基線)': f'{baseline_win_rate:.2f}%',
        '模型整體準確率 (Accuracy)': f'{accuracy:.2f}%',
        '高勝率組精確率 (Precision)': (
            f'{high_win_rate:.2f}%' if not np.isnan(high_win_rate) else 'N/A'
        ),
        '低勝率組實際勝率': (
            f'{low_win_rate:.2f}%' if not np.isnan(low_win_rate) else 'N/A'
        ),
        '模型召回率 (Recall)': f'{recall:.2f}%',
        'ROC AUC Score': f'{auc:.4f}' if not np.isnan(auc) else 'N/A',
    }])
    display(summary_metrics)

  # 7. 最新盤後預測
  print(f'\n=== 產生 {datetime.date.today()} 最新盤後預測建議 (Panel 模型) ===')
  latest_data = full_panel_df.groupby('Ticker').tail(1).copy()
  latest_data['Prob'] = model.predict_proba(latest_data[f_cols])[:, 1]

  pred_results = []
  for _, row in latest_data.iterrows():
    pred_results.append({
        '股票名稱': row['Name'],
        '股票代號': str(row['Ticker']).split('.')[0],
        '資料日期': row.name.strftime('%Y-%m-%d'),
        '預測勝率': f'{round(float(row["Prob"]) * 100, 2)}%',
        'raw_prob': float(row['Prob']),
    })

  pred_df = pd.DataFrame(pred_results).sort_values(
      by='raw_prob', ascending=False
  )
  triggered = pred_df[pred_df['raw_prob'] >= 0.52]

  if not triggered.empty:
    display(triggered[['股票名稱', '股票代號', '資料日期', '預測勝率']])
  else:
    print('今日無符合門檻之強勢標的。')


正在下載台股 (^TWII) 與美股 (^GSPC) 數據...
正在批次下載個股數據...
正在建構 Panel Data...

全市場 Panel 訓練集筆數: 203733 | 測試集筆數: 15627
正在訓練全市場通用 LightGBM 模型...

=== Panel Data 模型回測結果 (高勝率組：預測機率 >= 52%) ===
高勝率組訊號數: 438 | 成功數: 238 | 門檻內勝率 (Precision): 54.34%


Price,Ticker,Name,Target,Prob
Date,,,,
2026-04-28,8358.TWO,金居,1.0,0.650793
2026-04-29,8358.TWO,金居,1.0,0.650159
2026-04-08,3189.TW,景碩,1.0,0.644894
2026-04-08,6584.TWO,南俊國際,1.0,0.637353
2026-04-09,3363.TWO,上詮,1.0,0.629032
2026-05-04,6683.TWO,雍智科技,0.0,0.627881
2026-05-04,8021.TW,尖點,1.0,0.627144
2026-05-22,3189.TW,景碩,1.0,0.621870
2026-04-09,3234.TWO,光環,1.0,0.620855



=== Panel Data 模型回測結果 (低勝率組：預測機率 < 52%) ===
低勝率組訊號數: 15189 | 成功數: 7355 | 門檻外勝率: 48.42%


Price,Ticker,Name,Target,Prob
Date,,,,
2026-06-30,2342.TW,茂矽,0.0,0.519875
2026-04-20,3234.TWO,光環,0.0,0.519735
2026-05-13,3016.TW,嘉晶,0.0,0.519601
2026-04-10,2313.TW,華通,0.0,0.519415
2026-04-15,6683.TWO,雍智科技,0.0,0.519414
2026-06-01,3026.TW,禾伸堂,0.0,0.519296
2026-04-15,3167.TW,大量,0.0,0.519296
2026-05-22,3026.TW,禾伸堂,1.0,0.519296
2026-05-27,3026.TW,禾伸堂,1.0,0.519296



=== 全市場測試集整體統計與模型分類效能 ===


,測試集總樣本數,歷史自然勝率 (基線),模型整體準確率 (Accuracy),高勝率組精確率 (Precision),低勝率組實際勝率,模型召回率 (Recall),ROC AUC Score
0,15627,48.59%,51.65%,54.34%,48.42%,3.13%,0.5151



=== 產生 2026-08-17 最新盤後預測建議 (Panel 模型) ===


,股票名稱,股票代號,資料日期,預測勝率
124,環宇-KY,4991,2026-08-17,59.52%
112,華星光,4979,2026-08-17,58.34%
116,聯鈞,3450,2026-08-17,58.03%
115,波若威,3163,2026-08-17,55.88%
104,晶豪科,3006,2026-08-17,54.6%
242,德宏,5475,2026-08-17,54.36%
243,大量,3167,2026-08-17,53.37%
100,聯茂,6213,2026-08-17,53.3%
111,鈺創,5351,2026-08-17,53.11%
207,世紀,5314,2026-08-17,52.69%
